# OrthoFoundation DINOv3 Embeddings -> Hierarchical MIL

Trains Window MIL -> Series MIL -> `512 -> 128 -> 12` head directly on **precomputed,
frozen OrthoFoundation DINOv3 embeddings**. No DICOMs, no ResNet, no new feature
extraction. A trainable `Linear(DINO_DIM, 512)` projection (+ activation + norm)
adapts the DINOv3 dimensionality to the 512-D input the MIL stack expects.

**A note on the embedding file inspection this task asked for:** I was not able to
browse or download the actual contents of the
`prithvirajrp/orthofoundation-rsna-knee-raw-embeddings` Kaggle dataset from this
environment (Kaggle dataset pages render client-side and I have no Kaggle
credentials/API access here) — so I could not confirm the exact file format/keys
ahead of time the way I did for the in-chat-uploaded notebooks in earlier stages.
To compensate, **Section 4 below inspects the real files live, first thing, when you
run this on Kaggle**, printing extensions found, raw keys, nested types and shapes.
Section 5's loader is written to recognize several plausible layouts (nested
`dict[study][series]->array`, a flat `dict[study]->array`, or a single-record `dict`
with explicit `StudyInstanceUID`/`embeddings`/`series_ids`/`window_order` keys) and
is isolated in one function (`parse_container`) so it's easy to adjust if the printed
structure doesn't match any of those cases.


## 1. Imports

In [19]:
import os
import re
import json
import glob
import random
import pickle
from datetime import datetime,timezone

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm


## 2. Config

In [2]:

CONFIG = {
    # Data
    "EMBEDDINGS_DIR": "/kaggle/input/datasets/prithvirajrp/orthofoundation-rsna-knee-raw-embeddings",
    "LABELS_CSV": "/kaggle/input/datasets/saisahanas/rsna-knee-radiologist-report-labels/final_pseudo_labels.csv",
    "ID_COL": "StudyInstanceUID",
    "NUM_CLASSES": 12,
    "PROJECTED_DIM": 512,

    # Full-data split
    "SPLIT_SEED": 42,
    "TRAIN_FRACTION": 0.80,

    # Training
    "BATCH_SIZE": 16,
    "NUM_EPOCHS": 20,
    "EARLY_STOPPING_PATIENCE": 3,
    "LR": 1e-3,
    "WEIGHT_DECAY": 1e-4,
    "MAX_GRAD_NORM": 1.0,
    "SHUFFLE_SEED": 1337,

    # Model
    "ATTENTION_HIDDEN_DIM": 128,
    "HEAD_HIDDEN_DIM": 256,

    # Fresh initialization
    "INIT_FROM_STAGE1": False,
    "STAGE1_SEARCH_DIRS": ["/kaggle/working/STAGE1_OUTPUTS", "/kaggle/input", "/kaggle/working"],
    "STAGE1_WINDOW_MIL_FILENAME": "STAGE1_window_mil.pt",
    "STAGE1_SERIES_MIL_FILENAME": "STAGE1_series_mil.pt",
    "STAGE1_HEAD_FILENAME": "STAGE1_head.pt",

    # Checkpointing
    "CHECKPOINT_EVERY_N_STUDIES": 100,
    "OUTPUT_DIR": "/kaggle/working/ORTHOFOUNDATION_DINO_MIL_FULL_3750",
}

os.makedirs(CONFIG["OUTPUT_DIR"], exist_ok=True)

LATEST_CKPT_PATH = os.path.join(
    CONFIG["OUTPUT_DIR"], "ORTHO_DINO_MIL_latest_checkpoint.pt"
)
BEST_CKPT_PATH = os.path.join(
    CONFIG["OUTPUT_DIR"], "ORTHO_DINO_MIL_best_checkpoint.pt"
)
FINAL_MODEL_PATH = os.path.join(
    CONFIG["OUTPUT_DIR"], "ORTHO_DINO_MIL_final_model.pt"
)
HISTORY_CSV_PATH = os.path.join(
    CONFIG["OUTPUT_DIR"], "ORTHO_DINO_MIL_training_history.csv"
)
CONFIG_JSON_PATH = os.path.join(
    CONFIG["OUTPUT_DIR"], "ORTHO_DINO_MIL_config.json"
)

with open(CONFIG_JSON_PATH, "w") as f:
    json.dump(CONFIG, f, indent=2)

print("Output directory:", CONFIG["OUTPUT_DIR"])


Output directory: /kaggle/working/ORTHOFOUNDATION_DINO_MIL_FULL_3750


## 3. Device / AMP setup

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE_TYPE = device.type
USE_AMP = torch.cuda.is_available()

print("CUDA available:", torch.cuda.is_available())
print("Device:", device)
print("AMP enabled:", USE_AMP)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print(f"Total GPU memory: {props.total_memory / (1024**3):.2f} GB")
    print(f"Free/allocated at startup: "
          f"{torch.cuda.memory_reserved() / (1024**3):.2f} GB reserved, "
          f"{torch.cuda.memory_allocated() / (1024**3):.2f} GB allocated")


CUDA available: True
Device: cuda
AMP enabled: True
GPU: Tesla T4
Total GPU memory: 14.56 GB
Free/allocated at startup: 0.00 GB reserved, 0.00 GB allocated


## 4. Inspect the actual embedding files (run before writing any loading assumptions)

Finds every file under `EMBEDDINGS_DIR` regardless of extension, groups by extension,
and prints the raw structure of a sample file per extension: top-level type, dict
keys (if any), nesting, and tensor/array shapes. **Read this cell's output before
trusting Section 5's loader** — if the real layout doesn't match any case the loader
recognizes, `parse_container` in Section 5 is the one place to adapt.


In [4]:
def find_all_files(root):
    all_files = []
    for dirpath, _, filenames in os.walk(root):
        for fn in filenames:
            all_files.append(os.path.join(dirpath, fn))
    return all_files


all_embedding_files = find_all_files(CONFIG["EMBEDDINGS_DIR"])
print(f"Total files found under {CONFIG['EMBEDDINGS_DIR']}: {len(all_embedding_files)}")

by_ext = {}
for p in all_embedding_files:
    ext = os.path.splitext(p)[1].lower()
    by_ext.setdefault(ext, []).append(p)

print("\nFiles by extension:")
for ext, paths in sorted(by_ext.items(), key=lambda kv: -len(kv[1])):
    print(f"  {ext or '(no extension)'}: {len(paths)} file(s)  e.g. {paths[0]}")

if not all_embedding_files:
    raise FileNotFoundError(
        f"No files found under {CONFIG['EMBEDDINGS_DIR']!r}. Update CONFIG['EMBEDDINGS_DIR'] "
        "to the actual mount path for this dataset in your Kaggle environment."
    )


Total files found under /kaggle/input/datasets/prithvirajrp/orthofoundation-rsna-knee-raw-embeddings: 15

Files by extension:
  .npy 11: 1 file(s)  e.g. /kaggle/input/datasets/prithvirajrp/orthofoundation-rsna-knee-raw-embeddings/features.npy 11
  .npy 12: 1 file(s)  e.g. /kaggle/input/datasets/prithvirajrp/orthofoundation-rsna-knee-raw-embeddings/features.npy 12
  .npy 14: 1 file(s)  e.g. /kaggle/input/datasets/prithvirajrp/orthofoundation-rsna-knee-raw-embeddings/features.npy 14
  .npy: 1 file(s)  e.g. /kaggle/input/datasets/prithvirajrp/orthofoundation-rsna-knee-raw-embeddings/features.npy
  .npy 5: 1 file(s)  e.g. /kaggle/input/datasets/prithvirajrp/orthofoundation-rsna-knee-raw-embeddings/features.npy 5
  .npy 3: 1 file(s)  e.g. /kaggle/input/datasets/prithvirajrp/orthofoundation-rsna-knee-raw-embeddings/features.npy 3
  .npy 6: 1 file(s)  e.g. /kaggle/input/datasets/prithvirajrp/orthofoundation-rsna-knee-raw-embeddings/features.npy 6
  .npy 10: 1 file(s)  e.g. /kaggle/input/datas

In [5]:
def _describe(obj, depth=0, max_depth=2, max_items=3):
    indent = "  " * depth
    if depth > max_depth:
        print(f"{indent}...(truncated)")
        return

    if isinstance(obj, dict):
        keys = list(obj.keys())
        print(f"{indent}dict with {len(keys)} keys. Example keys: {keys[:max_items]} (key type: {type(keys[0]) if keys else None})")
        for k in keys[:max_items]:
            print(f"{indent}key {k!r} ->")
            _describe(obj[k], depth + 1, max_depth, max_items)
    elif isinstance(obj, (list, tuple)):
        print(f"{indent}{type(obj).__name__} of length {len(obj)}")
        if len(obj) > 0:
            _describe(obj[0], depth + 1, max_depth, max_items)
    elif isinstance(obj, np.ndarray):
        print(f"{indent}np.ndarray shape={obj.shape} dtype={obj.dtype}")
    elif torch.is_tensor(obj):
        print(f"{indent}torch.Tensor shape={tuple(obj.shape)} dtype={obj.dtype}")
    else:
        print(f"{indent}{type(obj)} -> {repr(obj)[:120]}")


def load_raw(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".npz":
        with np.load(path, allow_pickle=True) as npz:
            keys = list(npz.keys())
            key = "features" if "features" in keys else keys[0]
            raw = npz[key]
            return raw.item() if raw.shape == () else raw, keys
    elif ext in (".pt", ".pth"):
        return torch.load(path, map_location="cpu"), None
    elif ext == ".pkl":
        with open(path, "rb") as f:
            return pickle.load(f), None
    elif ext == ".npy":
        return np.load(path, allow_pickle=True), None
    else:
        raise ValueError(f"Unsupported extension for inspection: {ext}")


for ext, paths in by_ext.items():
    if ext not in (".npz", ".pt", ".pth", ".pkl", ".npy"):
        print(f"Skipping inspection of unsupported extension {ext!r} (file: {paths[0]})")
        continue
    print("=" * 70)
    print(f"Sample file for extension {ext!r}: {paths[0]}")
    try:
        obj, npz_keys = load_raw(paths[0])
        if npz_keys is not None:
            print("  npz top-level keys:", npz_keys)
        print(f"  Unwrapped python type: {type(obj)}")
        _describe(obj)
    except Exception as e:
        print(f"  Could not inspect this file: {e}")


Skipping inspection of unsupported extension '.npy 11' (file: /kaggle/input/datasets/prithvirajrp/orthofoundation-rsna-knee-raw-embeddings/features.npy 11)
Skipping inspection of unsupported extension '.npy 12' (file: /kaggle/input/datasets/prithvirajrp/orthofoundation-rsna-knee-raw-embeddings/features.npy 12)
Skipping inspection of unsupported extension '.npy 14' (file: /kaggle/input/datasets/prithvirajrp/orthofoundation-rsna-knee-raw-embeddings/features.npy 14)
Sample file for extension '.npy': /kaggle/input/datasets/prithvirajrp/orthofoundation-rsna-knee-raw-embeddings/features.npy
  Unwrapped python type: <class 'numpy.ndarray'>
np.ndarray shape=() dtype=object
Skipping inspection of unsupported extension '.npy 5' (file: /kaggle/input/datasets/prithvirajrp/orthofoundation-rsna-knee-raw-embeddings/features.npy 5)
Skipping inspection of unsupported extension '.npy 3' (file: /kaggle/input/datasets/prithvirajrp/orthofoundation-rsna-knee-raw-embeddings/features.npy 3)
Skipping inspectio

## 5. Generic, adaptable embeddings loader

Recognizes:
- a nested `dict[study_id -> dict[series_id -> array]]` (same layout used for the
  ResNet embeddings in earlier stages),
- a flat `dict[study_id -> array]` (treated as one pseudo-series per study),
- a single-record `dict` carrying explicit `StudyInstanceUID`/`embeddings`/
  `series_ids`/`window_order` keys (one file per study or per series), in which case
  windows are sorted by `window_order` if present, and split into series by
  `series_ids` if present.

If none of these match what Section 4 printed, adjust `parse_container` only —
everything downstream consumes its output and doesn't care how it got there.


In [6]:

def _to_str_key(x):
    if isinstance(x, bytes):
        return x.decode("utf-8", errors="ignore")
    return str(x)


def _as_window_array(x):
    if torch.is_tensor(x):
        x = x.detach().cpu().numpy()

    if isinstance(x, dict):
        raise ValueError(
            "Expected a feature array but found a nested dictionary."
        )

    arr = np.asarray(x, dtype=np.float32)

    if arr.ndim == 1:
        arr = arr[None, :]

    if arr.ndim != 2:
        raise ValueError(
            f"Expected [num_windows, embedding_dim], got {arr.shape}"
        )

    if arr.shape[0] == 0:
        return None

    return arr


def load_embeddings_index(files):
    merged = {}

    for file_idx, path in enumerate(files, 1):
        print(f"\n[{file_idx}/{len(files)}] Loading {os.path.basename(path)}")

        # Load .npy directly. This also handles names like features.npy 11,
        # which os.path.splitext() does not identify as having extension .npy.
        raw = np.load(path, allow_pickle=True)
        obj = raw.item() if raw.shape == () else raw

        if not isinstance(obj, dict):
            raise ValueError(
                f"Expected a dictionary in {path}, got {type(obj)}"
            )

        file_studies = 0

        for study_key, study_value in obj.items():
            study_id = _to_str_key(study_key)

            if not isinstance(study_value, dict):
                raise ValueError(
                    f"Expected series dictionary for study {study_id}, "
                    f"got {type(study_value)}"
                )

            if study_id not in merged:
                merged[study_id] = {}

            for series_key, feature_value in study_value.items():
                series_id = _to_str_key(series_key)
                arr = _as_window_array(feature_value)

                if arr is not None:
                    merged[study_id][series_id] = arr

            file_studies += 1

        print(f"  Studies loaded: {file_studies}")

    print("\n" + "=" * 60)
    print("EMBEDDING MERGE COMPLETE")
    print("=" * 60)
    print("Files merged:", len(files))
    print("Total studies:", len(merged))

    return merged


embeddings_index = load_embeddings_index(all_embedding_files)

# Verify actual embedding dimensions and grouping
sample_dims = set()
series_counts = []
window_counts = []

for study_id, series_dict in list(embeddings_index.items())[:100]:
    series_counts.append(len(series_dict))

    for arr in series_dict.values():
        sample_dims.add(arr.shape[-1])
        window_counts.append(arr.shape[0])

print("Embedding dimensions:", sample_dims)

if len(sample_dims) != 1:
    raise ValueError(f"Inconsistent or missing embedding dimensions: {sample_dims}")

DINO_DIM = next(iter(sample_dims))
print("DINO_DIM:", DINO_DIM)

if series_counts:
    print("Series/study — min:", min(series_counts),
          "max:", max(series_counts),
          "mean:", np.mean(series_counts))

if window_counts:
    print("Windows/series — min:", min(window_counts),
          "max:", max(window_counts),
          "mean:", np.mean(window_counts))



[1/15] Loading features.npy 11
  Studies loaded: 250

[2/15] Loading features.npy 12
  Studies loaded: 250

[3/15] Loading features.npy 14
  Studies loaded: 250

[4/15] Loading features.npy
  Studies loaded: 250

[5/15] Loading features.npy 5
  Studies loaded: 250

[6/15] Loading features.npy 3
  Studies loaded: 250

[7/15] Loading features.npy 6
  Studies loaded: 250

[8/15] Loading features.npy 10
  Studies loaded: 250

[9/15] Loading features.npy 7
  Studies loaded: 250

[10/15] Loading features.npy 13
  Studies loaded: 250

[11/15] Loading features.npy 15
  Studies loaded: 250

[12/15] Loading features.npy 4
  Studies loaded: 250

[13/15] Loading features.npy 2
  Studies loaded: 250

[14/15] Loading features.npy 9
  Studies loaded: 250

[15/15] Loading features.npy 8
  Studies loaded: 250

EMBEDDING MERGE COMPLETE
Files merged: 15
Total studies: 3750
Embedding dimensions: {1024}
DINO_DIM: 1024
Series/study — min: 4 max: 11 mean: 5.65
Windows/series — min: 9 max: 158 mean: 30.89203

## 6. Labels, matched to embeddings by StudyInstanceUID

In [7]:
labels_df_raw = pd.read_csv(CONFIG["LABELS_CSV"])

ID_COL = CONFIG["ID_COL"]
label_cols = [c for c in labels_df_raw.columns if c != ID_COL]

print("Detected label columns:", label_cols)
if len(label_cols) != CONFIG["NUM_CLASSES"]:
    print(f"WARNING: expected {CONFIG['NUM_CLASSES']} label columns, found {len(label_cols)}.")
    CONFIG["NUM_CLASSES"] = len(label_cols)

labels_df_raw[ID_COL] = labels_df_raw[ID_COL].astype(str)
labels_df = labels_df_raw.set_index(ID_COL)


def get_label_and_mask(study_id):
    row = labels_df.loc[study_id, label_cols].values.astype(np.float32)
    mask = (~np.isnan(row)).astype(np.float32)
    target = np.nan_to_num(row, nan=0.0)
    return target, mask


Detected label columns: ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']


## 7. The 1,000-study subset and the fixed deterministic 850/150 split

Same recipe/seed as the ResNet fine-tuning stage: `np.random.default_rng(42)`,
shuffle the study-id list in its original CSV row order, first 850 = train, last 150
= val. Matched against the embeddings index and the labels CSV; an explicit
leakage check confirms train and val never overlap.


In [8]:

ID_COL = CONFIG["ID_COL"]

labeled_ids = set(labels_df.index.astype(str))
embedded_ids = set(embeddings_index.keys())

# Use all studies that have both embeddings and labels
all_usable_ids = sorted(embedded_ids & labeled_ids)

# Deterministic shuffle and split
rng = np.random.default_rng(CONFIG["SPLIT_SEED"])
shuffled_ids = np.array(all_usable_ids, dtype=object)
rng.shuffle(shuffled_ids)

n_train = int(len(shuffled_ids) * CONFIG["TRAIN_FRACTION"])

train_ids = shuffled_ids[:n_train].tolist()
val_ids = shuffled_ids[n_train:].tolist()

print("Studies with embeddings:", len(embedded_ids))
print("Studies with labels:", len(labeled_ids))
print("Usable matched studies:", len(all_usable_ids))
print("Train studies:", len(train_ids))
print("Validation studies:", len(val_ids))

assert len(set(train_ids) & set(val_ids)) == 0
assert len(train_ids) + len(val_ids) == len(all_usable_ids)

# Save the exact split for reproducibility
pd.DataFrame({ID_COL: train_ids}).to_csv(
    os.path.join(CONFIG["OUTPUT_DIR"], "ORTHO_DINO_MIL_train_ids.csv"),
    index=False
)
pd.DataFrame({ID_COL: val_ids}).to_csv(
    os.path.join(CONFIG["OUTPUT_DIR"], "ORTHO_DINO_MIL_val_ids.csv"),
    index=False
)

print("No train/validation leakage.")


Studies with embeddings: 3750
Studies with labels: 4407
Usable matched studies: 3750
Train studies: 3000
Validation studies: 750
No train/validation leakage.


## 8. Batch construction (per-batch padding + masking, not whole-dataset padding)

Each mini-batch pads only to the largest series/window counts within that batch, with
explicit masks so padding never reaches attention or the loss. Embeddings are used
exactly as loaded — no truncation, padding-to-a-fixed-size, or reshaping of the
feature dimension itself.


In [9]:
def collate_items(items, embed_dim):
    B = len(items)
    max_series = max(max(len(it[1]), 1) for it in items)
    max_windows = max(max((arr.shape[0] for arr in it[1]), default=1) for it in items)
    max_windows = max(max_windows, 1)

    window_features = np.zeros((B, max_series, max_windows, embed_dim), dtype=np.float32)
    window_mask = np.zeros((B, max_series, max_windows), dtype=np.float32)
    series_mask = np.zeros((B, max_series), dtype=np.float32)
    targets = np.zeros((B, len(items[0][2])), dtype=np.float32)
    label_masks = np.zeros((B, len(items[0][3])), dtype=np.float32)
    study_ids = []

    for i, (study_id, series_arrays, target, label_mask) in enumerate(items):
        study_ids.append(study_id)
        targets[i] = target
        label_masks[i] = label_mask
        for s, arr in enumerate(series_arrays):
            n_win = arr.shape[0]
            window_features[i, s, :n_win, :] = arr
            window_mask[i, s, :n_win] = 1.0
            series_mask[i, s] = 1.0

    return {
        "study_ids": study_ids,
        "window_features": torch.from_numpy(window_features),
        "window_mask": torch.from_numpy(window_mask),
        "series_mask": torch.from_numpy(series_mask),
        "targets": torch.from_numpy(targets),
        "label_masks": torch.from_numpy(label_masks),
    }


def build_batch(study_id_batch, embeddings_index, embed_dim):
    items = []
    n_no_embedding = 0
    for sid in study_id_batch:
        series_dict = embeddings_index.get(sid, {})
        series_arrays = [arr for arr in series_dict.values() if arr is not None and arr.shape[0] > 0]
        if len(series_arrays) == 0:
            n_no_embedding += 1
        target, mask = get_label_and_mask(sid)
        items.append((sid, series_arrays, target, mask))
    return collate_items(items, embed_dim), n_no_embedding


## 9. Model: projection + Window/Series Attention MIL (Stage 1 architecture) + head

`MaskedAttentionMIL` and the head are structurally identical to Stage 1's (same
`attn_V`/`attn_U`/`attn_w` names, same bare `Linear->ReLU->Linear` head), so Stage 1
weights load directly when `CONFIG["INIT_FROM_STAGE1"]` is `True`.


In [10]:
class MaskedAttentionMIL(nn.Module):
    def __init__(self, in_dim=512, hidden_dim=128):
        super().__init__()
        self.attn_V = nn.Linear(in_dim, hidden_dim)
        self.attn_U = nn.Linear(in_dim, hidden_dim)
        self.attn_w = nn.Linear(hidden_dim, 1)

    def forward(self, x, mask):
        A_V = torch.tanh(self.attn_V(x))
        A_U = torch.sigmoid(self.attn_U(x))
        scores = self.attn_w(A_V * A_U).squeeze(-1)

        neg_inf = torch.finfo(scores.dtype).min
        scores = scores.masked_fill(mask == 0, neg_inf)

        all_masked = mask.sum(dim=-1, keepdim=True) == 0
        scores = torch.where(all_masked.expand_as(scores), torch.zeros_like(scores), scores)

        attn = torch.softmax(scores, dim=-1)
        attn = attn * mask
        denom = attn.sum(dim=-1, keepdim=True).clamp(min=1e-8)
        attn = attn / denom

        pooled = torch.sum(attn.unsqueeze(-1) * x, dim=-2)
        return pooled, attn


class EmbeddingProjection(nn.Module):
    """Linear(DINO_DIM, 512) + GELU + LayerNorm(512). GELU pairs well with
    transformer-derived (DINO) features; LayerNorm stabilizes the scale difference
    between DINOv3 and the ResNet-derived distribution the MIL stack originally saw,
    which matters most if Stage 1 weights are reused (Option A). Shape-preserving
    over all leading dimensions -- only the last (feature) dim changes."""

    def __init__(self, in_dim, out_dim=512):
        super().__init__()
        self.proj = nn.Linear(in_dim, out_dim)
        self.act = nn.GELU()
        self.norm = nn.LayerNorm(out_dim)

    def forward(self, x):
        return self.norm(self.act(self.proj(x)))


class OrthoDinoMILModel(nn.Module):
    def __init__(self, dino_dim, projected_dim, attn_hidden, head_hidden, num_classes):
        super().__init__()
        self.projection = EmbeddingProjection(dino_dim, projected_dim)
        self.window_mil = MaskedAttentionMIL(projected_dim, attn_hidden)
        self.series_mil = MaskedAttentionMIL(projected_dim, attn_hidden)
        self.head = nn.Sequential(
            nn.Linear(projected_dim, head_hidden),
            nn.ReLU(inplace=True),
            nn.Linear(head_hidden, num_classes),
        )

    def forward(self, window_features, window_mask, series_mask):
        projected = self.projection(window_features)                    # [B,S,W,512]
        series_vecs, _ = self.window_mil(projected, window_mask)        # [B,S,512]
        study_vec, _ = self.series_mil(series_vecs, series_mask)        # [B,512]
        logits = self.head(study_vec)                                   # [B,12]
        return logits


assert DINO_DIM is not None, "Could not infer DINO_DIM from the embeddings -- check Section 4/5 output."

model = OrthoDinoMILModel(
    DINO_DIM, CONFIG["PROJECTED_DIM"], CONFIG["ATTENTION_HIDDEN_DIM"], CONFIG["HEAD_HIDDEN_DIM"], CONFIG["NUM_CLASSES"]
).to(device)

print(model)


OrthoDinoMILModel(
  (projection): EmbeddingProjection(
    (proj): Linear(in_features=1024, out_features=512, bias=True)
    (act): GELU(approximate='none')
    (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  )
  (window_mil): MaskedAttentionMIL(
    (attn_V): Linear(in_features=512, out_features=128, bias=True)
    (attn_U): Linear(in_features=512, out_features=128, bias=True)
    (attn_w): Linear(in_features=128, out_features=1, bias=True)
  )
  (series_mil): MaskedAttentionMIL(
    (attn_V): Linear(in_features=512, out_features=128, bias=True)
    (attn_U): Linear(in_features=512, out_features=128, bias=True)
    (attn_w): Linear(in_features=128, out_features=1, bias=True)
  )
  (head): Sequential(
    (0): Linear(in_features=512, out_features=256, bias=True)
    (1): ReLU(inplace=True)
    (2): Linear(in_features=256, out_features=12, bias=True)
  )
)


## 10. Optional Stage 1 initialization (Option A) vs. fresh init (Option B, default)

In [11]:
def find_stage1_file(filename, search_dirs):
    for d in search_dirs:
        direct = os.path.join(d, filename)
        if os.path.exists(direct):
            return direct
        matches = glob.glob(os.path.join(d, "**", filename), recursive=True)
        if matches:
            return matches[0]
    return None


if CONFIG["INIT_FROM_STAGE1"]:
    window_mil_path = find_stage1_file(CONFIG["STAGE1_WINDOW_MIL_FILENAME"], CONFIG["STAGE1_SEARCH_DIRS"])
    series_mil_path = find_stage1_file(CONFIG["STAGE1_SERIES_MIL_FILENAME"], CONFIG["STAGE1_SEARCH_DIRS"])
    head_path = find_stage1_file(CONFIG["STAGE1_HEAD_FILENAME"], CONFIG["STAGE1_SEARCH_DIRS"])

    missing = [n for n, p in [("window_mil", window_mil_path), ("series_mil", series_mil_path), ("head", head_path)] if p is None]
    if missing:
        raise FileNotFoundError(
            f"INIT_FROM_STAGE1=True but could not find Stage 1 weights for: {missing}. "
            f"Searched: {CONFIG['STAGE1_SEARCH_DIRS']}"
        )

    try:
        model.window_mil.load_state_dict(torch.load(window_mil_path, map_location=device))
        model.series_mil.load_state_dict(torch.load(series_mil_path, map_location=device))
        model.head.load_state_dict(torch.load(head_path, map_location=device))
        print("Option A: loaded Stage 1 weights into Window MIL, Series MIL, and head.")
        print(f"  window_mil <- {window_mil_path}")
        print(f"  series_mil <- {series_mil_path}")
        print(f"  head       <- {head_path}")
    except Exception as e:
        raise RuntimeError(
            f"Found Stage 1 weight files but they did not load (architecture/dim mismatch?): {e}"
        )
else:
    print("Option B (default/primary experiment): Window MIL, Series MIL, and head start from fresh "
          "(PyTorch default) initialization. The projection layer is always freshly initialized "
          "regardless of this setting, since Stage 1 had no projection layer to reuse.")


Option B (default/primary experiment): Window MIL, Series MIL, and head start from fresh (PyTorch default) initialization. The projection layer is always freshly initialized regardless of this setting, since Stage 1 had no projection layer to reuse.


## 11. Loss, optimizer, AMP scaler

In [12]:
def masked_bce_loss(logits, targets, label_masks):
    loss_per_elem = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    loss_per_elem = loss_per_elem * label_masks
    denom = label_masks.sum().clamp(min=1.0)
    return loss_per_elem.sum() / denom


optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["LR"], weight_decay=CONFIG["WEIGHT_DECAY"])
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

print("Optimizer: AdamW, lr=", CONFIG["LR"], "weight_decay=", CONFIG["WEIGHT_DECAY"])


Optimizer: AdamW, lr= 0.001 weight_decay= 0.0001


/tmp/ipykernel_58/3548259758.py:9: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)


## 12. Sanity checks (run before training)

In [13]:
print("=" * 70)
print("SANITY CHECKS")
print("=" * 70)

print(f"Embedding files found:                 {len(all_embedding_files)}")
print(f"Studies with embeddings:                {len(embedded_ids)}")
print(f"Matched labeled studies (train+val):    {len(train_ids) + len(val_ids)}")
print(f"Train / Val sizes:                      {len(train_ids)} / {len(val_ids)}")
print(f"DINOv3 embedding dimension:              {DINO_DIM}")
print(f"Projection output dimension:             {CONFIG['PROJECTED_DIM']}")

series_counts, window_counts = [], []
shape_anomalies = []
for sid in (train_ids + val_ids):
    sdict = embeddings_index.get(sid, {})
    series_counts.append(len(sdict))
    for arr in sdict.values():
        window_counts.append(arr.shape[0])
        if arr.shape[-1] != DINO_DIM:
            shape_anomalies.append((sid, arr.shape))

if series_counts:
    print(f"Series per study:   min={min(series_counts)} max={max(series_counts)} mean={np.mean(series_counts):.2f}")
if window_counts:
    print(f"Windows per series: min={min(window_counts)} max={max(window_counts)} mean={np.mean(window_counts):.2f}")

if shape_anomalies:
    print(f"WARNING: {len(shape_anomalies)} series have an embedding dim != {DINO_DIM}. First few: {shape_anomalies[:5]}")
else:
    print("All matched studies have the expected embedding dimensionality.")

n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model total parameters:                  {n_params:,}")
print(f"Model trainable parameters:              {n_trainable:,}")

print(f"CUDA available:                          {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:                                      {torch.cuda.get_device_name(0)}")
    print(f"Free CUDA memory (approx):                "
          f"{(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_reserved()) / (1024**3):.2f} GB")


SANITY CHECKS
Embedding files found:                 15
Studies with embeddings:                3750
Matched labeled studies (train+val):    3750
Train / Val sizes:                      3000 / 750
DINOv3 embedding dimension:              1024
Projection output dimension:             512
Series per study:   min=3 max=14 mean=5.53
Windows per series: min=9 max=318 mean=31.56
All matched studies have the expected embedding dimensionality.
Model total parameters:                  923,150
Model trainable parameters:              923,150
CUDA available:                          True
GPU:                                      Tesla T4
Free CUDA memory (approx):                14.54 GB


In [14]:
def check_finite(model, sample_logits=None):
    ok = True
    for name, p in model.named_parameters():
        if not torch.isfinite(p).all():
            print(f"WARNING: parameter {name} contains non-finite values!")
            ok = False
    if sample_logits is not None and not torch.isfinite(sample_logits).all():
        print("WARNING: sample forward output contains non-finite values!")
        ok = False
    return ok


sanity_batch, _ = build_batch(train_ids[:4], embeddings_index, DINO_DIM)
wf = sanity_batch["window_features"].to(device)
wm = sanity_batch["window_mask"].to(device)
sm = sanity_batch["series_mask"].to(device)

model.eval()
with torch.no_grad():
    sanity_logits = model(wf, wm, sm)

print("Sanity forward pass output shape:", tuple(sanity_logits.shape))
assert sanity_logits.shape[-1] == CONFIG["NUM_CLASSES"], "Model did not output exactly 12 logits!"
print("Confirmed: model outputs exactly", CONFIG["NUM_CLASSES"], "logits per study.")

finite_ok = check_finite(model, sanity_logits)
print("All parameters and sample outputs finite:", finite_ok)
assert finite_ok, "Non-finite values detected before training even started -- investigate before proceeding."

model.train()
print("=" * 70)


Sanity forward pass output shape: (4, 12)
Confirmed: model outputs exactly 12 logits per study.
All parameters and sample outputs finite: True


## 13. Checkpoint utilities

Checkpoints contain: projection/Window MIL/Series MIL/head weights, optimizer and
AMP scaler state, epoch and in-epoch batch index, best validation AUC and its full
history, the fixed train/val study-ID lists, and RNG states (python/numpy/torch/cuda)
for best-effort reproducibility on resume.


In [20]:
def get_rng_states():
    state = {"python": random.getstate(), "numpy": np.random.get_state(), "torch": torch.get_rng_state()}
    if torch.cuda.is_available():
        state["torch_cuda"] = torch.cuda.get_rng_state_all()
    return state


def set_rng_states(state):
    try:
        random.setstate(state["python"])
        np.random.set_state(state["numpy"])
        torch.set_rng_state(state["torch"])
        if torch.cuda.is_available() and "torch_cuda" in state:
            torch.cuda.set_rng_state_all(state["torch_cuda"])
    except Exception as e:
        print("Could not fully restore RNG state (non-fatal):", e)


def save_checkpoint(path, epoch, resume_batch_index, val_auc, best_val_auc, val_auc_history):
    ckpt = {
        "projection_state_dict": model.projection.state_dict(),
        "window_mil_state_dict": model.window_mil.state_dict(),
        "series_mil_state_dict": model.series_mil.state_dict(),
        "head_state_dict": model.head.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "epoch": epoch,
        "resume_batch_index": resume_batch_index,
        "val_auc": val_auc,
        "best_val_auc": best_val_auc,
        "val_auc_history": val_auc_history,
        "train_ids": train_ids,
        "val_ids": val_ids,
        "rng_states": get_rng_states(),
        "config": CONFIG,
        "dino_dim": DINO_DIM,
        "saved_at": datetime.now(timezone.utc).isoformat(),
    }
    torch.save(ckpt, path)


def load_checkpoint(path):
    ckpt = torch.load(path, map_location=device, weights_only=False)
    model.projection.load_state_dict(ckpt["projection_state_dict"])
    model.window_mil.load_state_dict(ckpt["window_mil_state_dict"])
    model.series_mil.load_state_dict(ckpt["series_mil_state_dict"])
    model.head.load_state_dict(ckpt["head_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scaler.load_state_dict(ckpt["scaler_state_dict"])
    if "rng_states" in ckpt:
        set_rng_states(ckpt["rng_states"])

    if ckpt.get("train_ids") != train_ids or ckpt.get("val_ids") != val_ids:
        print("WARNING: the train/val split stored in this checkpoint does not match the split "
              "just computed in this session. Continuing with the freshly computed split, but "
              "this should not happen if CONFIG/SAMPLE_CSV are unchanged.")

    return ckpt["epoch"], ckpt["resume_batch_index"], ckpt["val_auc"], ckpt["best_val_auc"], ckpt["val_auc_history"]


## 14. Resume logic

In [16]:
start_epoch = 0
start_batch_index = 0
best_val_auc = -1.0
last_val_auc = 0.0
val_auc_history = []

if os.path.exists(LATEST_CKPT_PATH):
    start_epoch, start_batch_index, last_val_auc, best_val_auc, val_auc_history = load_checkpoint(LATEST_CKPT_PATH)
    print(
        f"Resumed from checkpoint: epoch={start_epoch}, batch_index={start_batch_index}, "
        f"best_val_auc={best_val_auc:.4f}, {len(val_auc_history)} prior epoch(s) recorded"
    )
else:
    print("No checkpoint found. Starting fresh from epoch 0.")


No checkpoint found. Starting fresh from epoch 0.


## 15. Validation (mean + per-class ROC-AUC, NaN-safe for single-class columns)

In [17]:
@torch.no_grad()
def run_validation(val_ids, batch_size):
    model.eval()
    all_probs, all_targets, all_masks = [], [], []

    for start in range(0, len(val_ids), batch_size):
        batch_ids = val_ids[start:start + batch_size]
        batch, _ = build_batch(batch_ids, embeddings_index, DINO_DIM)

        wf = batch["window_features"].to(device, non_blocking=True)
        wm = batch["window_mask"].to(device, non_blocking=True)
        sm = batch["series_mask"].to(device, non_blocking=True)

        with torch.autocast(device_type=DEVICE_TYPE, dtype=torch.float16, enabled=USE_AMP):
            logits = model(wf, wm, sm)

        all_probs.append(torch.sigmoid(logits.float()).cpu().numpy())
        all_targets.append(batch["targets"].numpy())
        all_masks.append(batch["label_masks"].numpy())

    all_probs = np.concatenate(all_probs, axis=0)
    all_targets = np.concatenate(all_targets, axis=0)
    all_masks = np.concatenate(all_masks, axis=0)

    per_class_auc = {}
    for c, col_name in enumerate(label_cols):
        valid = all_masks[:, c] > 0
        y_true = all_targets[valid, c]
        y_score = all_probs[valid, c]
        if len(np.unique(y_true)) < 2:
            per_class_auc[col_name] = float("nan")  # reported, excluded from the mean below
            continue
        try:
            per_class_auc[col_name] = roc_auc_score(y_true, y_score)
        except Exception:
            per_class_auc[col_name] = float("nan")

    auc_values = list(per_class_auc.values())
    finite_aucs = [a for a in auc_values if not np.isnan(a)]
    mean_auc = float(np.mean(finite_aucs)) if finite_aucs else 0.0

    model.train()
    return mean_auc, per_class_auc


## 16. Training loop

Mini-batch padded/masked training (Section 8), AdamW, gradient clipping at
`max_norm=1.0`, AMP, checkpointing every `CHECKPOINT_EVERY_N_STUDIES` studies and at
every epoch boundary, validation + per-class AUC + training history logging every
epoch, early stopping on validation mean ROC-AUC.


In [21]:
def append_history_row(epoch, train_loss, val_auc, lr):
    row = pd.DataFrame([{
        "epoch": epoch,
        "train_loss": train_loss,
        "val_mean_auc": val_auc,
        "learning_rate": lr,
        "timestamp": datetime.utcnow().isoformat(),
    }])
    if os.path.exists(HISTORY_CSV_PATH):
        row.to_csv(HISTORY_CSV_PATH, mode="a", header=False, index=False)
    else:
        row.to_csv(HISTORY_CSV_PATH, mode="w", header=True, index=False)


def train():
    global start_epoch, start_batch_index, best_val_auc, last_val_auc, val_auc_history

    epochs_without_improvement = 0
    model.train()

    for epoch in range(start_epoch, CONFIG["NUM_EPOCHS"]):
        order = train_ids.copy()
        random.Random(CONFIG["SHUFFLE_SEED"] + epoch).shuffle(order)
        batches = [order[i:i + CONFIG["BATCH_SIZE"]] for i in range(0, len(order), CONFIG["BATCH_SIZE"])]

        batch_index_start = start_batch_index if epoch == start_epoch else 0
        studies_since_last_ckpt = 0
        epoch_loss_sum, epoch_loss_count = 0.0, 0

        pbar = tqdm(range(batch_index_start, len(batches)), desc=f"Epoch {epoch}",
                    initial=batch_index_start, total=len(batches))

        for batch_index in pbar:
            batch_ids = batches[batch_index]
            batch, n_missing = build_batch(batch_ids, embeddings_index, DINO_DIM)

            wf = batch["window_features"].to(device, non_blocking=True)
            wm = batch["window_mask"].to(device, non_blocking=True)
            sm = batch["series_mask"].to(device, non_blocking=True)
            targets = batch["targets"].to(device, non_blocking=True)
            label_masks = batch["label_masks"].to(device, non_blocking=True)

            optimizer.zero_grad()
            with torch.autocast(device_type=DEVICE_TYPE, dtype=torch.float16, enabled=USE_AMP):
                logits = model(wf, wm, sm)
                loss = masked_bce_loss(logits, targets, label_masks)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["MAX_GRAD_NORM"])
            scaler.step(optimizer)
            scaler.update()

            epoch_loss_sum += loss.item()
            epoch_loss_count += 1
            studies_since_last_ckpt += len(batch_ids)
            pbar.set_postfix(loss=epoch_loss_sum / max(epoch_loss_count, 1))

            if studies_since_last_ckpt >= CONFIG["CHECKPOINT_EVERY_N_STUDIES"]:
                save_checkpoint(LATEST_CKPT_PATH, epoch, batch_index + 1, last_val_auc, best_val_auc, val_auc_history)
                print(f"\nCheckpoint saved ({studies_since_last_ckpt} studies since last) -> {LATEST_CKPT_PATH}")
                studies_since_last_ckpt = 0

        mean_train_loss = epoch_loss_sum / max(epoch_loss_count, 1)
        current_lr = optimizer.param_groups[0]["lr"]

        val_auc, per_class_auc = run_validation(val_ids, CONFIG["BATCH_SIZE"])
        last_val_auc = val_auc
        val_auc_history.append(val_auc)

        print(f"\nEpoch {epoch} | train_loss={mean_train_loss:.4f} | val_mean_auc={val_auc:.4f} | lr={current_lr:.2e}")
        for col_name, auc in per_class_auc.items():
            print(f"  {col_name:30s} AUC: {auc if not np.isnan(auc) else 'NaN (single class)'}")

        append_history_row(epoch, mean_train_loss, val_auc, current_lr)

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            epochs_without_improvement = 0
            save_checkpoint(BEST_CKPT_PATH, epoch + 1, 0, val_auc, best_val_auc, val_auc_history)
            print(f"New best model (val AUC {best_val_auc:.4f}) -> {BEST_CKPT_PATH}")
        else:
            epochs_without_improvement += 1

        save_checkpoint(LATEST_CKPT_PATH, epoch + 1, 0, val_auc, best_val_auc, val_auc_history)
        start_batch_index = 0

        if epochs_without_improvement >= CONFIG["EARLY_STOPPING_PATIENCE"]:
            print(f"\nEarly stopping: no improvement in validation mean AUC for "
                  f"{CONFIG['EARLY_STOPPING_PATIENCE']} epochs.")
            break

    print("\nTraining complete.")
    print(f"Best validation mean ROC-AUC: {best_val_auc:.4f}")


train()


Epoch 0:   0%|          | 0/188 [00:00<?, ?it/s]


Checkpoint saved (112 studies since last) -> /kaggle/working/ORTHOFOUNDATION_DINO_MIL_FULL_3750/ORTHO_DINO_MIL_latest_checkpoint.pt

Checkpoint saved (112 studies since last) -> /kaggle/working/ORTHOFOUNDATION_DINO_MIL_FULL_3750/ORTHO_DINO_MIL_latest_checkpoint.pt

Checkpoint saved (112 studies since last) -> /kaggle/working/ORTHOFOUNDATION_DINO_MIL_FULL_3750/ORTHO_DINO_MIL_latest_checkpoint.pt

Checkpoint saved (112 studies since last) -> /kaggle/working/ORTHOFOUNDATION_DINO_MIL_FULL_3750/ORTHO_DINO_MIL_latest_checkpoint.pt

Checkpoint saved (112 studies since last) -> /kaggle/working/ORTHOFOUNDATION_DINO_MIL_FULL_3750/ORTHO_DINO_MIL_latest_checkpoint.pt

Checkpoint saved (112 studies since last) -> /kaggle/working/ORTHOFOUNDATION_DINO_MIL_FULL_3750/ORTHO_DINO_MIL_latest_checkpoint.pt

Checkpoint saved (112 studies since last) -> /kaggle/working/ORTHOFOUNDATION_DINO_MIL_FULL_3750/ORTHO_DINO_MIL_latest_checkpoint.pt

Checkpoint saved (112 studies since last) -> /kaggle/working/ORTHOFO

/tmp/ipykernel_58/1825395478.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat(),


Epoch 1:   0%|          | 0/188 [00:00<?, ?it/s]


Checkpoint saved (112 studies since last) -> /kaggle/working/ORTHOFOUNDATION_DINO_MIL_FULL_3750/ORTHO_DINO_MIL_latest_checkpoint.pt

Checkpoint saved (112 studies since last) -> /kaggle/working/ORTHOFOUNDATION_DINO_MIL_FULL_3750/ORTHO_DINO_MIL_latest_checkpoint.pt

Checkpoint saved (112 studies since last) -> /kaggle/working/ORTHOFOUNDATION_DINO_MIL_FULL_3750/ORTHO_DINO_MIL_latest_checkpoint.pt

Checkpoint saved (112 studies since last) -> /kaggle/working/ORTHOFOUNDATION_DINO_MIL_FULL_3750/ORTHO_DINO_MIL_latest_checkpoint.pt

Checkpoint saved (112 studies since last) -> /kaggle/working/ORTHOFOUNDATION_DINO_MIL_FULL_3750/ORTHO_DINO_MIL_latest_checkpoint.pt

Checkpoint saved (112 studies since last) -> /kaggle/working/ORTHOFOUNDATION_DINO_MIL_FULL_3750/ORTHO_DINO_MIL_latest_checkpoint.pt

Checkpoint saved (112 studies since last) -> /kaggle/working/ORTHOFOUNDATION_DINO_MIL_FULL_3750/ORTHO_DINO_MIL_latest_checkpoint.pt

Checkpoint saved (112 studies since last) -> /kaggle/working/ORTHOFO

/tmp/ipykernel_58/1825395478.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat(),


... (50 outputs left out)

/tmp/ipykernel_58/1825395478.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat(),


Epoch 19:   0%|          | 0/188 [00:00<?, ?it/s]


Checkpoint saved (112 studies since last) -> /kaggle/working/ORTHOFOUNDATION_DINO_MIL_FULL_3750/ORTHO_DINO_MIL_latest_checkpoint.pt

Checkpoint saved (112 studies since last) -> /kaggle/working/ORTHOFOUNDATION_DINO_MIL_FULL_3750/ORTHO_DINO_MIL_latest_checkpoint.pt

Checkpoint saved (112 studies since last) -> /kaggle/working/ORTHOFOUNDATION_DINO_MIL_FULL_3750/ORTHO_DINO_MIL_latest_checkpoint.pt

Checkpoint saved (112 studies since last) -> /kaggle/working/ORTHOFOUNDATION_DINO_MIL_FULL_3750/ORTHO_DINO_MIL_latest_checkpoint.pt

Checkpoint saved (112 studies since last) -> /kaggle/working/ORTHOFOUNDATION_DINO_MIL_FULL_3750/ORTHO_DINO_MIL_latest_checkpoint.pt

Checkpoint saved (112 studies since last) -> /kaggle/working/ORTHOFOUNDATION_DINO_MIL_FULL_3750/ORTHO_DINO_MIL_latest_checkpoint.pt

Checkpoint saved (112 studies since last) -> /kaggle/working/ORTHOFOUNDATION_DINO_MIL_FULL_3750/ORTHO_DINO_MIL_latest_checkpoint.pt

Checkpoint saved (112 studies since last) -> /kaggle/working/ORTHOFO

/tmp/ipykernel_58/1825395478.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat(),


## 17. Final verification: reload the best checkpoint into a fresh model instance


In [22]:
print("Loading best checkpoint and evaluating on the fixed validation set...")

final_model = OrthoDinoMILModel(
    DINO_DIM, CONFIG["PROJECTED_DIM"], CONFIG["ATTENTION_HIDDEN_DIM"], CONFIG["HEAD_HIDDEN_DIM"], CONFIG["NUM_CLASSES"]
).to(device)

best_ckpt = torch.load(BEST_CKPT_PATH, map_location=device, weights_only=False)
final_model.projection.load_state_dict(best_ckpt["projection_state_dict"])
final_model.window_mil.load_state_dict(best_ckpt["window_mil_state_dict"])
final_model.series_mil.load_state_dict(best_ckpt["series_mil_state_dict"])
final_model.head.load_state_dict(best_ckpt["head_state_dict"])
final_model.eval()

print(f"Loaded best checkpoint from epoch index {best_ckpt['epoch'] - 1} "
      f"(stored best_val_auc={best_ckpt['best_val_auc']:.4f})")


Loading best checkpoint and evaluating on the fixed validation set...
Loaded best checkpoint from epoch index 16 (stored best_val_auc=0.8168)


In [23]:
@torch.no_grad()
def evaluate_with_model(eval_model, val_ids, batch_size):
    all_probs, all_targets, all_masks = [], [], []
    for start in range(0, len(val_ids), batch_size):
        batch_ids = val_ids[start:start + batch_size]
        batch, _ = build_batch(batch_ids, embeddings_index, DINO_DIM)
        wf = batch["window_features"].to(device)
        wm = batch["window_mask"].to(device)
        sm = batch["series_mask"].to(device)
        logits = eval_model(wf, wm, sm)
        all_probs.append(torch.sigmoid(logits.float()).cpu().numpy())
        all_targets.append(batch["targets"].numpy())
        all_masks.append(batch["label_masks"].numpy())

    all_probs = np.concatenate(all_probs, axis=0)
    all_targets = np.concatenate(all_targets, axis=0)
    all_masks = np.concatenate(all_masks, axis=0)

    per_class_auc = {}
    for c, col_name in enumerate(label_cols):
        valid = all_masks[:, c] > 0
        y_true = all_targets[valid, c]
        y_score = all_probs[valid, c]
        if len(np.unique(y_true)) < 2:
            per_class_auc[col_name] = float("nan")
            continue
        try:
            per_class_auc[col_name] = roc_auc_score(y_true, y_score)
        except Exception:
            per_class_auc[col_name] = float("nan")

    finite_aucs = [a for a in per_class_auc.values() if not np.isnan(a)]
    mean_auc = float(np.mean(finite_aucs)) if finite_aucs else 0.0
    return mean_auc, per_class_auc


final_mean_auc, final_per_class_auc = evaluate_with_model(final_model, val_ids, CONFIG["BATCH_SIZE"])

print("=" * 70)
print("FINAL VALIDATION RESULTS (best checkpoint, freshly loaded model)")
print("=" * 70)
print(f"Mean validation ROC-AUC: {final_mean_auc:.4f}")
for col_name, auc in final_per_class_auc.items():
    print(f"  {col_name:30s} AUC: {auc if not np.isnan(auc) else 'NaN (single class)'}")


FINAL VALIDATION RESULTS (best checkpoint, freshly loaded model)
Mean validation ROC-AUC: 0.8168
  ACL                            AUC: 0.8486661367471308
  MCL                            AUC: 0.7300594142699406
  Medial Meniscus                AUC: 0.8487292036562109
  Lateral Meniscus               AUC: 0.7353913072799001
  Medial OA                      AUC: 0.8634202551400004
  Lateral OA                     AUC: 0.8299923029905146
  PF OA                          AUC: 0.8095168315219603
  Effusion                       AUC: 0.8293404822413686
  Synovitis                      AUC: 0.860033364715987
  Baker's                        AUC: 0.7860558353297805
  Contusion                      AUC: 0.8407762450315641
  Fracture                       AUC: 0.8191471430979269


## 18. Save the complete final model and demonstrate inference on one study

In [24]:
torch.save({
    "projection_state_dict": final_model.projection.state_dict(),
    "window_mil_state_dict": final_model.window_mil.state_dict(),
    "series_mil_state_dict": final_model.series_mil.state_dict(),
    "head_state_dict": final_model.head.state_dict(),
    "dino_dim": DINO_DIM,
    "config": CONFIG,
    "final_val_mean_auc": final_mean_auc,
    "final_per_class_auc": final_per_class_auc,
}, FINAL_MODEL_PATH)
print("Saved complete final model to:", FINAL_MODEL_PATH)


Saved complete final model to: /kaggle/working/ORTHOFOUNDATION_DINO_MIL_FULL_3750/ORTHO_DINO_MIL_final_model.pt


In [25]:
# Demonstrate loading the saved model in a completely fresh instance
demo_model = OrthoDinoMILModel(
    DINO_DIM, CONFIG["PROJECTED_DIM"], CONFIG["ATTENTION_HIDDEN_DIM"], CONFIG["HEAD_HIDDEN_DIM"], CONFIG["NUM_CLASSES"]
).to(device)

demo_ckpt = torch.load(FINAL_MODEL_PATH, map_location=device)
demo_model.projection.load_state_dict(demo_ckpt["projection_state_dict"])
demo_model.window_mil.load_state_dict(demo_ckpt["window_mil_state_dict"])
demo_model.series_mil.load_state_dict(demo_ckpt["series_mil_state_dict"])
demo_model.head.load_state_dict(demo_ckpt["head_state_dict"])
demo_model.eval()

demo_study_id = val_ids[0]
demo_batch, _ = build_batch([demo_study_id], embeddings_index, DINO_DIM)

with torch.no_grad():
    demo_logits = demo_model(
        demo_batch["window_features"].to(device),
        demo_batch["window_mask"].to(device),
        demo_batch["series_mask"].to(device),
    )
    demo_probs = torch.sigmoid(demo_logits).cpu().numpy()[0]

print(f"Inference demo on study {demo_study_id} using the freshly-loaded saved model:")
for col_name, prob in zip(label_cols, demo_probs):
    print(f"  {col_name:30s} p={prob:.4f}")


UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL numpy._core.multiarray.scalar was not an allowed global by default. Please use `torch.serialization.add_safe_globals([numpy._core.multiarray.scalar])` or the `torch.serialization.safe_globals([numpy._core.multiarray.scalar])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn 

## 19. Final summary and output file sizes

In [ ]:
print("=" * 70)
print("ORTHOFOUNDATION DINO MIL -- RUN SUMMARY")
print("=" * 70)
print(f"Best validation mean ROC-AUC: {best_val_auc:.4f}")
print(f"Final (reloaded) validation mean ROC-AUC: {final_mean_auc:.4f}")

print("\nValidation AUC history (by epoch):")
for i, auc in enumerate(val_auc_history):
    print(f"  Epoch {i}: {auc:.4f}")

print("\nOutput files:")
output_paths = [
    LATEST_CKPT_PATH,
    BEST_CKPT_PATH,
    FINAL_MODEL_PATH,
    HISTORY_CSV_PATH,
    CONFIG_JSON_PATH,
    os.path.join(CONFIG["OUTPUT_DIR"], "ORTHO_DINO_MIL_train_ids.csv"),
    os.path.join(CONFIG["OUTPUT_DIR"], "ORTHO_DINO_MIL_val_ids.csv"),
]
for p in output_paths:
    if os.path.exists(p):
        size_mb = os.path.getsize(p) / (1024**2)
        print(f"  {p}  ({size_mb:.2f} MB)")
    else:
        print(f"  {p}  -- MISSING")

print(
    "\nReminder: /kaggle/working is NOT permanently stored. Download "
    f"{CONFIG['OUTPUT_DIR']} now, or use Kaggle's 'Save Version -> Save & Run All' "
    "so these files are attached as a notebook Output before the session ends."
)
